# K-Means Clustering: Complete Guide

Welcome to this comprehensive module on K-Means Clustering. K-Means is one of the most popular and straightforward unsupervised machine learning algorithms used for partitioning a dataset into a set of k groups (or clusters).

## What we will cover:
- The fundamental theory behind K-Means
- Implementing K-Means from scratch
- Utilizing Scikit-Learn for efficient clustering
- Methods to determine the optimal number of clusters (Elbow Method & Silhouette Score)
- The critical importance of Feature Scaling
- Limitations of K-Means and alternative approaches
- Practical implementation on a simulated real-world dataset

In [ ]:
# 1. Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Machine Learning libraries
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Set display options for DataFrames
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

# Set global plotting style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook", font_scale=1.2)

# Set random seed for reproducibility across the notebook
np.random.seed(42)
print("Libraries successfully imported and random seed set to 42.")

## 1. Introduction to K-Means Clustering

Unsupervised learning involves training a model on data that is neither classified nor labeled. The algorithm must discover patterns, structures, or relationships within the data on its own.

**K-Means** is a centroid-based clustering algorithm. Its goal is to divide n observations into k clusters, where each observation belongs to the cluster with the nearest mean (centroid). 

### Real-world Applications:
- **Customer Segmentation:** Grouping customers by purchasing behavior.
- **Image Compression:** Reducing the number of colors in an image.
- **Anomaly Detection:** Identifying outliers that do not belong to any dense cluster.

In [ ]:
# Generate basic synthetic data for our initial demonstrations
print("Generating synthetic data with 4 distinct clusters...")

# make_blobs generates isotropic Gaussian blobs for clustering
X, y_true = make_blobs(
    n_samples=400, 
    centers=4, 
    cluster_std=0.80, 
    random_state=42
)

# Convert to a pandas DataFrame for easier viewing
df_blobs = pd.DataFrame(X, columns=["Feature_1", "Feature_2"])
df_blobs["True_Label"] = y_true

# Display the first few rows and basic info
print("\nDataset Preview:")
print(df_blobs.head())
print("\nDataset Statistics:")
print(df_blobs.describe())

# Visualize the raw, unlabeled data (how the algorithm sees it)
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, color="gray", alpha=0.6)
plt.title("Raw Unlabeled Data (Input to K-Means)", fontsize=14)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Core Concept 1: The Algorithm Step-by-Step

The standard K-Means algorithm (often called Lloyd's algorithm) works in an iterative manner:

1. **Initialization:** Randomly select K data points to be the initial cluster centers (centroids).
2. **Assignment:** Calculate the distance from each point to each centroid. Assign each point to the closest centroid.
3. **Update:** Recalculate the centroids by taking the mean of all data points assigned to that centroid's cluster.
4. **Repeat:** Repeat steps 2 and 3 until the centroids no longer move significantly (convergence).

Let's implement Step 1 (Initialization) manually to see it in action.

In [ ]:
# Step 1: Manual Initialization
k = 4

# Randomly pick k indices from our dataset
random_indices = np.random.choice(X.shape[0], size=k, replace=False)
initial_centroids = X[random_indices]

print(f"Randomly selected {k} initial centroids:")
print(initial_centroids)

# Plot the data along with our random initial centroids
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, color="gray", alpha=0.5, label="Data Points")
plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1], 
            c="red", s=200, marker="X", edgecolor="black", 
            label="Initial Centroids")
plt.title("Step 1: Random Initialization of Centroids")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2 & 3: Assignment and Update

Now we will run the full iterative loop. We calculate distances using the standard Euclidean distance formula: 
Distance = square_root( sum( (point_features - centroid_features)^2 ) )

Watch how the centroids shift towards the center of mass of their assigned points.

In [ ]:
def run_manual_kmeans(X, k, max_iters=10):
    """A simple manual implementation of K-Means."""
    # 1. Initialize centroids randomly
    idx = np.random.choice(X.shape[0], size=k, replace=False)
    centroids = X[idx]
    
    for iteration in range(max_iters):
        # 2. Assignment Step
        # Calculate distances from each point to each centroid
        # We use a list comprehension and numpy broadcasting for speed
        distances = np.array([np.linalg.norm(X - c, axis=1) for c in centroids])
        
        # Assign each point to the closest centroid (index of minimum distance)
        labels = np.argmin(distances, axis=0)
        
        # 3. Update Step
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        
        # Check for convergence (if centroids stop moving)
        if np.all(centroids == new_centroids):
            print(f"Converged at iteration {iteration}!")
            break
            
        centroids = new_centroids
        
    return labels, centroids

# Run our manual function
labels_manual, centroids_manual = run_manual_kmeans(X, k=4)

# Visualize the final result
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=labels_manual, cmap="viridis", s=30, alpha=0.6)
plt.scatter(centroids_manual[:, 0], centroids_manual[:, 1], 
            c="red", s=200, marker="X", edgecolor="black", 
            label="Final Centroids")
plt.title("Manual K-Means Final Result")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Core Concept 2: Using Scikit-Learn

While it is great to build algorithms from scratch for learning, in practice, we use optimized libraries like `scikit-learn`.

Scikit-learn's `KMeans` class is highly optimized and includes several enhancements, such as `k-means++` initialization, which smartly selects initial centroids to speed up convergence and avoid poor local optima.

In [ ]:
print("Initializing Scikit-Learn KMeans model...")
kmeans = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)

# Fit the model and predict labels
sklearn_labels = kmeans.fit_predict(X)
sklearn_centroids = kmeans.cluster_centers_

print("Model fitted successfully.")
print("Centroid Coordinates:")
print(np.round(sklearn_centroids, 2))

# Visualize Scikit-Learn's results
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=sklearn_labels, cmap="plasma", s=30, alpha=0.6)
plt.scatter(sklearn_centroids[:, 0], sklearn_centroids[:, 1], 
            c="cyan", s=250, marker="*", edgecolor="black", 
            label="Sklearn Centroids")
plt.title("Scikit-Learn KMeans Clustering", fontsize=14)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Core Concept 3: Choosing the Right K (Elbow Method)

In supervised learning, we know the number of classes. In unsupervised learning, we often don't know the optimal number of clusters (K). 

**The Elbow Method** helps us choose K by calculating the Within-Cluster Sum of Squares (WCSS) for different values of K. WCSS is the sum of squared distances between each point and its assigned centroid.
As K increases, WCSS decreases. We look for an "elbow" in the graph where the rate of decrease sharply slows down.

In [ ]:
# Calculate WCSS (Inertia) for different values of K
wcss = []
K_range = range(1, 11)

print("Calculating WCSS for K=1 to K=10...")
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(X)
    # inertia_ is the scikit-learn attribute for WCSS
    wcss.append(km.inertia_)

print("Calculation complete.")

# Plot the Elbow Graph
plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, marker="o", linestyle="--", color="b", linewidth=2, markersize=8)
plt.title("The Elbow Method For Optimal K", fontsize=15)
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("WCSS (Inertia)", fontsize=12)
plt.xticks(K_range)
plt.axvline(x=4, color="red", linestyle="dotted", label="Optimal K = 4")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

print("Notice the sharp bend (elbow) at K=4. Adding more clusters beyond 4 yields diminishing returns.")

## 5. Core Concept 4: Choosing K (Silhouette Score)

Sometimes the elbow is not very clear. The **Silhouette Score** is an alternative metric that measures how similar an object is to its own cluster (cohesion) compared to other clusters (separation).

- The score ranges from -1 to +1.
- A score near +1 indicates the point is far from neighboring clusters.
- A score near 0 indicates the point is on the boundary between clusters.
- A score near -1 indicates the point might be assigned to the wrong cluster.

We want to maximize the average silhouette score.

In [ ]:
silhouette_scores = []
K_range_sil = range(2, 11)  # Silhouette score requires at least 2 clusters

print("Calculating Silhouette Scores for K=2 to K=10...")
for k in K_range_sil:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"K={k} | Silhouette Score: {score:.4f}")

# Plot Silhouette Scores
plt.figure(figsize=(9, 5))
plt.plot(K_range_sil, silhouette_scores, marker="s", linestyle="-", color="green", linewidth=2)
plt.title("Silhouette Score vs Number of Clusters", fontsize=15)
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("Average Silhouette Score", fontsize=12)
plt.xticks(K_range_sil)
plt.axvline(x=4, color="red", linestyle="dotted", label="Maximum Score at K=4")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## 6. The k-means++ Initialization Technique

Standard K-means randomly initializes centroids. If two initial centroids fall into the same true cluster, the algorithm can get stuck in a bad local minimum.

`k-means++` solves this by placing the initial centroids far away from each other.
1. Pick the first centroid randomly.
2. Calculate distance from every point to the nearest chosen centroid.
3. Pick the next centroid with a probability proportional to that distance squared (points further away have a higher chance).
4. Repeat until K centroids are chosen.

Let's see if there is a difference in iterations required for convergence.

In [ ]:
# Comparing Random vs K-Means++ Initialization
km_random = KMeans(n_clusters=4, init="random", n_init=1, random_state=55)
km_plus = KMeans(n_clusters=4, init="k-means++", n_init=1, random_state=55)

km_random.fit(X)
km_plus.fit(X)

print("Performance Comparison:")
print(f"Random Initialization took {km_random.n_iter_} iterations to converge.")
print(f"K-Means++ Initialization took {km_plus.n_iter_} iterations to converge.")
print("\nWhile both eventually find the clusters, k-means++ usually requires fewer iterations and avoids bad local optima.")

## 7. Core Concept 5: The Critical Importance of Feature Scaling

K-Means relies entirely on Euclidean distance. If one feature has a much larger range of values than another (e.g., Salary in $100,000s vs Age in 10s), the larger feature will completely dominate the distance calculation.

Let's deliberately create a dataset with mismatched scales to see how K-Means fails without scaling.

In [ ]:
# Generate synthetic unscaled data
# Feature 1 ranges from 0 to 10 (e.g., years of experience)
# Feature 2 ranges from 20000 to 100000 (e.g., salary)
np.random.seed(42)
f1_c1 = np.random.normal(3, 1, 100)
f2_c1 = np.random.normal(30000, 5000, 100)

f1_c2 = np.random.normal(8, 1, 100)
f2_c2 = np.random.normal(80000, 5000, 100)

f1 = np.concatenate([f1_c1, f1_c2])
f2 = np.concatenate([f2_c1, f2_c2])

X_unscaled = np.column_stack((f1, f2))
df_unscaled = pd.DataFrame(X_unscaled, columns=["Experience_Years", "Salary"])

print("Unscaled Data Description:")
print(df_unscaled.describe())

# Run K-Means directly on unscaled data
km_unscaled = KMeans(n_clusters=2, n_init=10, random_state=42)
unscaled_labels = km_unscaled.fit_predict(X_unscaled)

plt.figure(figsize=(10, 5))
plt.scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=unscaled_labels, cmap="bwr", alpha=0.7)
plt.title("K-Means on UNSCALED Data (Clusters driven entirely by Salary)")
plt.xlabel("Experience (Years)")
plt.ylabel("Salary ($)")
plt.grid(alpha=0.3)
plt.show()

print("Notice how the clusters split horizontally. Experience_Years is completely ignored because its variance is tiny compared to Salary.")

## Fixing with StandardScaler

To fix this, we standard-scale our data so that every feature has a mean of 0 and a standard deviation of 1. 
Formula: z = (x - mean) / standard_deviation

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)

# Run K-Means on scaled data
km_scaled = KMeans(n_clusters=2, n_init=10, random_state=42)
scaled_labels = km_scaled.fit_predict(X_scaled)

# Plotting the before and after
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot Unscaled
ax1.scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=unscaled_labels, cmap="coolwarm", alpha=0.7)
ax1.set_title("Without Scaling: Clustering Ignores X-Axis")
ax1.set_xlabel("Experience (Years)")
ax1.set_ylabel("Salary ($)")
ax1.grid(alpha=0.3)

# Plot Scaled
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], c=scaled_labels, cmap="coolwarm", alpha=0.7)
ax2.set_title("With StandardScaler: Both Features Matter")
ax2.set_xlabel("Scaled Experience")
ax2.set_ylabel("Scaled Salary")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Rule of Thumb: ALWAYS scale your data before applying distance-based algorithms like K-Means.")

## 8. Common Pitfalls: Non-spherical Data

K-Means makes a strict assumption: Clusters are convex, spherical, and isotropic (they have roughly the same variance in all directions). 

If your data forms complex shapes like interlocking rings or moons, K-Means will fail completely, regardless of scaling or initializations.

In [ ]:
# Generate "Moons" dataset - two interlocking half-circles
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# Apply K-Means
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42)
moon_labels_km = km_moons.fit_predict(X_moons)

plt.figure(figsize=(8, 6))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_labels_km, cmap="viridis", alpha=0.7)
plt.scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1], 
            c="red", marker="X", s=200, label="Centroids")
plt.title("K-Means Failure: Non-Spherical Data")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("K-Means literally slices the data down the middle because it relies solely on central points.")

## Alternative: DBSCAN

For complex shapes, density-based algorithms like **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) are superior. DBSCAN connects dense regions of points together, regardless of their overall geometric shape.

In [ ]:
# Apply DBSCAN to the moons dataset
# eps is the maximum distance between two samples to be considered in the same neighborhood
dbscan = DBSCAN(eps=0.2, min_samples=5)
moon_labels_db = dbscan.fit_predict(X_moons)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# K-Means Plot
ax1.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_labels_km, cmap="viridis", alpha=0.7)
ax1.set_title("K-Means (Fails to capture structure)")
ax1.grid(alpha=0.3)

# DBSCAN Plot
ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_labels_db, cmap="viridis", alpha=0.7)
ax2.set_title("DBSCAN (Perfectly captures structure)")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Takeaway: Know the assumptions of your algorithm. K-Means is fast but assumes circular clusters.")

## 9. Practice Exercise: Customer Segmentation

Let's put everything together. We will create a synthetic "Customer Mall" dataset. 
It contains customer `Age`, `Annual_Income_k`, and `Spending_Score` (1-100).

In [ ]:
# Generating Mall Customer Data
np.random.seed(123)

# Cluster 1: High Income, High Spending (Target Customers)
inc_1 = np.random.normal(90, 10, 50)
spend_1 = np.random.normal(85, 10, 50)
age_1 = np.random.normal(32, 5, 50)

# Cluster 2: Low Income, High Spending (Careless)
inc_2 = np.random.normal(30, 8, 50)
spend_2 = np.random.normal(80, 10, 50)
age_2 = np.random.normal(25, 4, 50)

# Cluster 3: High Income, Low Spending (Frugal)
inc_3 = np.random.normal(85, 12, 50)
spend_3 = np.random.normal(25, 10, 50)
age_3 = np.random.normal(45, 8, 50)

# Cluster 4: Average Income, Average Spending (Standard)
inc_4 = np.random.normal(55, 10, 100)
spend_4 = np.random.normal(50, 10, 100)
age_4 = np.random.normal(50, 10, 100)

income = np.concatenate([inc_1, inc_2, inc_3, inc_4])
spending = np.concatenate([spend_1, spend_2, spend_3, spend_4])
age = np.concatenate([age_1, age_2, age_3, age_4])

df_customers = pd.DataFrame({
    "Age": age,
    "Annual_Income_k": income,
    "Spending_Score": spending
})

# Ensure bounds make sense
df_customers["Spending_Score"] = df_customers["Spending_Score"].clip(1, 100)
df_customers["Age"] = df_customers["Age"].clip(18, 80)

print("Customer Dataset Created. Total rows:", len(df_customers))
print(df_customers.head())

### Exercise Instructions:
1. Scale the dataset.
2. Use the Elbow method to find the optimal K (test K from 1 to 10).
3. Fit a K-Means model with the optimal K.
4. Assign the cluster labels back to the original dataframe.
5. Plot `Annual_Income_k` vs `Spending_Score` colored by cluster.

*Try writing the code yourself before looking at the solution cell below!*

In [ ]:
### SOLUTION ###

# 1. Scale the dataset
customer_scaler = StandardScaler()
X_cust_scaled = customer_scaler.fit_transform(df_customers)

# 2. Find optimal K using Elbow Method
wcss_cust = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, init="k-means++", n_init=10, random_state=42)
    km.fit(X_cust_scaled)
    wcss_cust.append(km.inertia_)

# Plotting Elbow
plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), wcss_cust, marker="o", linestyle="--")
plt.title("Elbow Method for Customer Data")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.grid(alpha=0.3)
plt.show()

print("Based on data generation and the elbow graph, optimal K=4.")

# 3. Fit K-Means with K=4
final_km = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)
cust_labels = final_km.fit_predict(X_cust_scaled)

# 4. Assign labels
df_customers["Customer_Segment"] = cust_labels

# 5. Visualize
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_customers, x="Annual_Income_k", y="Spending_Score", 
                hue="Customer_Segment", palette="tab10", s=100, alpha=0.8)
plt.title("Customer Segmentation Analysis", fontsize=16)
plt.xlabel("Annual Income ($k)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.grid(alpha=0.3)
plt.legend(title="Segment")
plt.tight_layout()
plt.show()

print("Excellent! The model successfully identified our 4 hidden customer personas.")

## 10. Summary and Key Takeaways

- **Unsupervised Nature:** K-Means finds hidden patterns without requiring labeled target variables.
- **The Mechanism:** Iterates between assigning points to the nearest centroid and updating centroid positions.
- **Evaluation Metrics:** Use the Elbow Method (looking for the bend in WCSS) or Silhouette Score to find the optimal number of clusters.
- **Scaling is Mandatory:** Always standardize your data before applying K-Means to prevent high-variance features from dominating.
- **Limitations:** Fails on non-spherical, complex datasets. Sensitive to outliers. Requires you to manually define K.
- **Initialization:** Always use `init="k-means++"` rather than random initialization to speed up convergence and find better optimal solutions.

In [ ]:
# Bonus: 3D Visualization of the Customer Dataset
# K-means works in n-dimensions. Here we visualize the 3D space.
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

colors = ["red", "blue", "green", "purple"]

for i in range(4):
    subset = df_customers[df_customers["Customer_Segment"] == i]
    ax.scatter(subset["Age"], subset["Annual_Income_k"], subset["Spending_Score"], 
               s=50, alpha=0.7, c=colors[i], label=f"Segment {i}")

ax.set_xlabel("Age")
ax.set_ylabel("Annual Income ($k)")
ax.set_zlabel("Spending Score")
ax.set_title("3D Customer Segmentation Space")
plt.legend()
plt.show()

print("Module Complete! You are now equipped to apply K-Means clustering to a variety of datasets.")